
# Project : Autonomous Research Agent (ReAct Loop + ArXiv Tool Calling)
**Domain:** Research / Literature Review | **Level:** Advanced | **Stack:** Python, Groq (gpt-oss-120b) function calling, ArXiv API

## Problem statement
A normal RAG pipeline does one search and one answer. Real research needs many steps: search, read, decide if it is enough, search again with different words. In this project the LLM is the decision maker. It calls tools (ArXiv search, save note) in a loop and finally writes a short literature review with research gaps.

The big risk with agents is infinite loops, so I added 3 safety guards: a max step limit, blocking repeated identical tool calls, and a forced final answer when the limit is reached.

## What I built
- Tool registry with 2 tools: `search_arxiv` and `save_note`
- ReAct style loop (think, call tool, read result, think again)
- Guard 1: max steps. Guard 2: repeated-call detection. Guard 3: forced final answer
- Full trace table of every step the agent took
- A stress test with a nonsense topic to see the guards working

## Step 0: Install libraries

In [1]:
!pip install -q openai pandas

## Step 1: Connect to Groq

In [2]:
import os, json, time, re
from getpass import getpass
from openai import OpenAI

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key here: ")

client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
MODEL = "openai/gpt-oss-120b"

def call_llm(prompt, system="You are a helpful assistant.", json_mode=False):
    for attempt in range(3):
        try:
            msgs = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
            if json_mode:
                res = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0, response_format={"type": "json_object"})
            else:
                res = client.chat.completions.create(model=MODEL, messages=msgs, temperature=0)
            return (res.choices[0].message.content or "").strip()
        except Exception as e:
            print("LLM error (try", attempt + 1, "):", str(e)[:150])
            time.sleep(6)
    return ""

print(call_llm("Say hello in 5 words"))

Paste your Groq API key here: ··········
Hello there, nice to meet!


## Step 2: Tools
The agent has 2 tools.
- `search_arxiv` calls the free ArXiv API and returns titles + short abstracts
- `save_note` is a scratchpad where the agent can store findings

If ArXiv is down or blocked the tool falls back to a small offline list of 5 real papers, so the notebook still runs. The trace will show when this happens.

In [3]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET

offline_papers = [
    {"title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", "year": "2020", "note": "Introduces RAG, mixing a retriever with a generator so the model can use outside documents."},
    {"title": "ReAct: Synergizing Reasoning and Acting in Language Models", "year": "2022", "note": "Shows LLMs can alternate between thinking steps and tool actions to solve tasks."},
    {"title": "Lost in the Middle: How Language Models Use Long Contexts", "year": "2023", "note": "Finds models use info at the start and end of the prompt better than the middle."},
    {"title": "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection", "year": "2023", "note": "Model decides when to retrieve and critiques its own output."},
    {"title": "Corrective Retrieval Augmented Generation", "year": "2024", "note": "Adds a retrieval evaluator that triggers web search when retrieved docs are poor."},
]

def search_arxiv(query, max_results=3):
    words = []
    for w in query.split():
        w = re.sub(r"[^A-Za-z0-9]", "", w)
        if w:
            words.append("all:" + w)
    if len(words) == 0:
        return "No papers found. Query was empty."
    search_q = " AND ".join(words)
    url = "https://export.arxiv.org/api/query?search_query=" + urllib.parse.quote(search_q) + "&start=0&max_results=" + str(max_results)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        data = urllib.request.urlopen(req, timeout=20).read()
        root = ET.fromstring(data)
        ns = "{http://www.w3.org/2005/Atom}"
        results = []
        for entry in root.findall(ns + "entry"):
            title = " ".join(entry.find(ns + "title").text.split())
            summary = " ".join(entry.find(ns + "summary").text.split())
            year = entry.find(ns + "published").text[:4]
            link = entry.find(ns + "id").text
            results.append("Title: " + title + "\nYear: " + year + "\nAbstract: " + summary[:450] + "\nLink: " + link)
        if len(results) == 0:
            return "No papers found for this query."
        return "\n---\n".join(results)
    except Exception as e:
        print("   (arxiv failed, using offline sample:", str(e)[:70], ")")
        results = []
        for p in offline_papers[:max_results]:
            results.append("Title: " + p["title"] + "\nYear: " + p["year"] + "\nAbstract: " + p["note"] + " [offline sample]")
        return "\n---\n".join(results)

notes_list = []

def save_note(note):
    notes_list.append(note)
    return "Note saved. Total notes: " + str(len(notes_list))

tool_functions = {"search_arxiv": search_arxiv, "save_note": save_note}

tool_defs = [
    {"type": "function", "function": {
        "name": "search_arxiv",
        "description": "Search ArXiv for research papers. Use short keyword queries.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "search keywords"},
            "max_results": {"type": "integer", "description": "number of papers, default 3"}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "save_note",
        "description": "Save a short note about a finding to the scratchpad.",
        "parameters": {"type": "object", "properties": {
            "note": {"type": "string", "description": "the finding to remember"}},
            "required": ["note"]}}},
]

print(search_arxiv("retrieval augmented generation", 2)[:600])

Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
Year: 2025
Abstract: We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrie
Link: http://arxiv.org/abs/2506.06962v3
---
Title: Aut


## Step 3: The agent loop
Each round the model either asks for tools or gives a final answer.

Guards:
1. `max_steps` stops the loop after N rounds
2. If the model repeats the exact same tool call, I don't run it again, I tell it to change the query or finish
3. When steps run out, I force one last call with no tools so it must write the answer

In [4]:
agent_system = '''You are a research assistant. Investigate the topic using the tools.
Do 2 or 3 searches with different keywords, save useful notes, then stop searching.
Final answer format:
## Key findings
## Papers used (title + year)
## Research gaps
Only use information that came from tool results.'''

def run_agent(topic, max_steps=6):
    messages = [{"role": "system", "content": agent_system}, {"role": "user", "content": topic}]
    seen_calls = set()
    trace = []
    blocked = 0

    for step in range(1, max_steps + 1):
        try:
            res = client.chat.completions.create(model=MODEL, messages=messages, tools=tool_defs, tool_choice="auto", temperature=0)
        except Exception as e:
            print("LLM error:", str(e)[:150])
            time.sleep(6)
            continue

        msg = res.choices[0].message

        if not msg.tool_calls:
            trace.append({"step": step, "action": "FINAL ANSWER", "args": "", "result_chars": len(msg.content or "")})
            return msg.content, trace, blocked

        call_list = []
        for tc in msg.tool_calls:
            call_list.append({"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}})
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": call_list})

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except Exception:
                args = {}
            key = name + json.dumps(args, sort_keys=True)

            if key in seen_calls:
                output = "You already made this exact call. Use different keywords or write the final answer now."
                blocked = blocked + 1
                action = name + " (BLOCKED repeat)"
            else:
                seen_calls.add(key)
                try:
                    output = tool_functions[name](**args)
                except Exception as e:
                    output = "Tool error: " + str(e)[:100]
                action = name

            trace.append({"step": step, "action": action, "args": json.dumps(args)[:70], "result_chars": len(output)})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    messages.append({"role": "user", "content": "Step limit reached. Write the final answer now using what you have. Do not call tools."})
    final = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    trace.append({"step": max_steps + 1, "action": "FORCED FINAL ANSWER", "args": "", "result_chars": len(final.choices[0].message.content or "")})
    return final.choices[0].message.content, trace, blocked

print("agent ready")

agent ready


## Step 4: Run it on a real topic

In [5]:
import pandas as pd
from IPython.display import Markdown, display

notes_list.clear()
topic = "How can retrieval augmented generation reduce hallucinations in large language models?"
answer, trace, blocked = run_agent(topic, max_steps=6)

trace_df = pd.DataFrame(trace)
print("Agent trace:")
print(trace_df)
print("\nNotes saved by agent:", len(notes_list))
print("Repeated calls blocked:", blocked)
display(Markdown(answer))

Agent trace:
   step               action  \
0     1         search_arxiv   
1     2         search_arxiv   
2     3            save_note   
3     4         search_arxiv   
4     5         search_arxiv   
5     6            save_note   
6     7  FORCED FINAL ANSWER   

                                                args  result_chars  
0  {"max_results": 5, "query": "retrieval augment...          3133  
1  {"max_results": 5, "query": "dynamic retrieval...          3043  
2  {"note": "RAG mitigates hallucinations by grou...            26  
3  {"max_results": 1, "query": "To Retrieve or No...            31  
4  {"max_results": 5, "query": "retrieval augment...          3014  
5  {"note": "Dynamic retrieval with uncertainty d...            26  
6                                                             5563  

Notes saved by agent: 2
Repeated calls blocked: 0


**## Key findings**

1. **Grounding generation in external evidence** – Retrieval‑augmented generation (RAG) equips a large language model (LLM) with a non‑parametric knowledge store. By conditioning the decoder on passages that have been retrieved from a curated corpus, the model can cite concrete facts instead of relying solely on its internal parameters, which are prone to outdated or spurious associations. This grounding is the primary mechanism by which hallucinations are reduced【note 1】.

2. **Retriever quality directly correlates with hallucination mitigation** – Comparative studies show that stronger retrievers (e.g., dense embeddings, hybrid BM25 + neural) retrieve more relevant documents, giving the generator higher‑quality evidence and consequently lowering the rate of factual errors. When the retrieved set contains the correct answer, the LLM’s probability of hallucinating drops dramatically【note 1】.

3. **Dynamic, confidence‑driven retrieval** – Instead of invoking retrieval for every token, recent work proposes to trigger the retrieval module only when the model’s internal uncertainty exceeds a threshold. Uncertainty can be estimated from token‑level entropy, self‑consistency checks, or a dedicated classifier. By retrieving only when needed, the system avoids over‑reliance on noisy or irrelevant documents while still providing factual support for low‑confidence regions, which is especially beneficial for long‑form question answering【note 2】.

4. **Post‑retrieval verification (entailment/consistency checks)** – After a passage is retrieved, a lightweight entailment model can verify whether the generated statement is supported by the evidence. If the check fails, the system can either request additional retrieval, re‑rank the evidence, or flag the output as potentially hallucinated. This “evidence‑aligned verification” layer further curtails hallucinations that arise from mis‑interpreting retrieved text【search result 3】.

5. **Fine‑tuning the generator on retrieved contexts** – Training (or instruction‑tuning) the LLM on examples where the input includes retrieved passages teaches the model to attend to the external evidence and to copy or paraphrase it faithfully. Fine‑tuned RAG models (e.g., Finetune‑RAG) show lower hallucination rates than vanilla LLMs even when the retriever occasionally returns partially irrelevant documents【search result 3】.

6. **Hybrid retrieval strategies** – Combining sparse (BM25) and dense (embedding‑based) retrieval, or augmenting with knowledge‑graph walks, improves coverage of both lexical and semantic matches. Hybrid pipelines retrieve a more diverse evidence set, reducing the chance that the generator will have to “fill gaps” with invented content【search result 1】.

7. **Mitigating adversarial corpus poisoning** – While not directly a hallucination issue, protecting the retrieval corpus from malicious injections (e.g., via retrieval‑stage defenses) is essential because poisoned documents can induce systematic hallucinations. Defensive filtering of retrieved candidates helps maintain factual integrity【search result 2】.

**## Papers used (title + year)**  

- *Hybrid Retrieval for Hallucination Mitigation in Large Language Models: A Comparative Analysis* (2025)  
- *To Retrieve or Not to Retrieve? Uncertainty Detection for Dynamic Retrieval Augmented Generation* (2025)  
- *RAGPart & RAGMask: Retrieval‑Stage Defenses Against Corpus Poisoning in Retrieval‑Augmented Generation* (2025)  
- *Finetune‑RAG: Fine‑Tuning Language Models to Resist Hallucination in Retrieval‑Augmented Generation* (2025)  
- *Evidence‑Aligned Entity Verification for Hallucination Detection in Retrieval‑Augmented Generation* (2026)  
- *FACTOID: FACtual enTailment fOr hallucInation Detection* (2024)  

**## Research gaps**

1. **Unified uncertainty estimation** – Existing dynamic‑retrieval methods use heterogeneous confidence signals (entropy, self‑consistency, classifier scores). A principled, model‑agnostic uncertainty estimator that works across tasks and modalities is still lacking.

2. **End‑to‑end training of retrieval‑generation‑verification loops** – Most pipelines train the retriever, generator, and verifier separately. Joint optimization that back‑propagates verification loss through the retriever could further tighten the grounding and reduce hallucinations.

3. **Scalable verification for long‑form generation** – Current entailment checks are applied sentence‑by‑sentence; scaling to multi‑paragraph or document‑level generation without prohibitive compute remains an open problem.

4. **Robustness to noisy or contradictory evidence** – When the corpus contains conflicting documents, the generator may still hallucinate by arbitrating incorrectly. Methods for conflict detection and resolution within RAG are under‑explored.

5. **Multimodal RAG for hallucination control** – While multimodal RAG (text + images/video) has been surveyed, systematic studies on how visual or auditory evidence influences factuality and hallucination rates are scarce.

6. **User‑controlled grounding** – Interfaces that let end‑users specify the desired evidence source or level of grounding (e.g., “cite only peer‑reviewed papers”) are not well studied, yet they could improve trust in high‑stakes applications.

7. **Long‑term knowledge updates** – Retrieval corpora need continual refreshing to keep up with evolving facts. Efficient mechanisms for incremental corpus updates that preserve retrieval quality and hallucination mitigation have not been fully addressed.

## Step 5: Stress test - what if searches return nothing?
I gave the agent a nonsense topic. This is where naive agents keep looping. Let's see how the guards handle it.

In [6]:
notes_list.clear()
answer2, trace2, blocked2 = run_agent("zzqxv flurbnak wibblegrom theory", max_steps=4)

print(pd.DataFrame(trace2))
print("\nrepeated calls blocked:", blocked2)
print("total steps used:", trace2[-1]["step"], "(limit was 4 + 1 forced answer)")
print("\nfinal answer preview:\n", (answer2 or "")[:500])

   step        action                                       args  result_chars
0     1  search_arxiv    {"max_results": 5, "query": "flurbnak"}            31
1     2  search_arxiv  {"max_results": 5, "query": "wibblegrom"}            31
2     3  FINAL ANSWER                                                      868

repeated calls blocked: 0
total steps used: 3 (limit was 4 + 1 forced answer)

final answer preview:
 ## Key findings
- No scholarly articles or pre‑prints were found on **“flurbnak”**, **“wibblegrom”**, or the combined phrase **“zzqxv flurbnak wibblegrom theory”** in the arXiv database.

## Papers used (title + year)
- *None* – the searches returned no matching results.

## Research gaps
- The absence of any indexed publications indicates that these terms do not correspond to an established scientific concept or theory in the current literature.  
- If these are emerging or highly specialized t


## Step 6: Save the report

In [7]:
with open("research_report.md", "w") as file:
    file.write("# Research report\n\nTopic: " + topic + "\n\n" + answer)
print("saved research_report.md")

search_steps = 0
for t in trace:
    if t["action"].startswith("search_arxiv"):
        search_steps = search_steps + 1

print()
print("Resume bullet (from my own run):")
print("Built an autonomous research agent using Groq function calling and the ArXiv API with a ReAct-style loop; it made", search_steps, "searches and", len(trace), "total steps on a live topic, and I added max-step, repeat-call blocking and forced-final-answer guards which I stress tested with a nonsense query.")

saved research_report.md

Resume bullet (from my own run):
Built an autonomous research agent using Groq function calling and the ArXiv API with a ReAct-style loop; it made 4 searches and 7 total steps on a live topic, and I added max-step, repeat-call blocking and forced-final-answer guards which I stress tested with a nonsense query.


## Key learnings
- The model decides what to do next, code only runs the tools and keeps the loop safe
- Agents can loop forever, so limits and repeat detection are a must, not a nice-to-have
- When passing messages back to the API, I had to rebuild the assistant message by hand with only id, name and arguments
- Truncating abstracts (450 chars) keeps token usage and rate limits under control
- Tools should fail gracefully (offline fallback) so one bad API call does not break the whole run

## Skills shown
Agentic AI, Function / tool calling, ReAct pattern, Loop safety, ArXiv API, XML parsing, Prompt design, Groq API

## Ideas to extend
- Rewrite the loop in LangGraph with a state object
- Add Semantic Scholar as a second tool
- Add a human approval step before saving the final report

---
**Built by Akshat Kesharwani** | Portfolio: https://akshatkesharwani-info.github.io/ | GitHub: https://github.com/akshatkesharwani-info

Project idea taken from the Dipanshu.Decoded Agentic AI handbook. I rewrote the code to run free on Groq and added my own test questions and evaluation.